Awesome. Tuning the heuristic and penalty weights is where you get to change the "personality" and efficiency of the agent. By adjusting these values, you're essentially changing the edge weights in the graph, forcing the algorithm to prioritize different types of safety or speed.

Here are a few experiments you can run directly in **Cell 2**.

### Experiment 1: Changing the Heuristic (The Distance Metric)

The heuristic `h(self, n)` tells the agent how to estimate the distance to the goal. You are currently using **Euclidean distance** (straight line). Since your agent can move in 8 directions (including diagonals), **Chebyshev distance** is often a mathematically tighter fit for grid-based 8-way movement.

Find this block in **Cell 2**:

```python
    def h(self, n):
        """Heuristic function: Euclidean distance."""
        return math.hypot(n.x - self.start.x, n.y - self.start.y)

```

**Try replacing it with Chebyshev Distance:**
This measures the maximum of the absolute differences along the axes. It assumes diagonal movement costs the same as cardinal movement (which aligns nicely with your base cost of `1.0` and `1.414`).

```python
    def h(self, n):
        """Heuristic function: Chebyshev distance."""
        dx = abs(n.x - self.start.x)
        dy = abs(n.y - self.start.y)
        return max(dx, dy) 

```

*Effect:* You might notice the algorithm expands slightly fewer nodes (lower `self.km` replanning events) because the heuristic doesn't underestimate diagonal distances quite as much as Euclidean does in grid setups.

---

### Experiment 2: The "Overly Cautious" Agent

Right now, your agent adds a +100 cost if a node is directly touching fire. We can expand this buffer to make the agent deeply afraid of even getting *near* the fire, forcing it to hug the walls.

In the `c(self, u, v)` function, look for the **FIRE PROXIMITY PENALTY**.

**Modify it to penalize smoke heavily:**

```python
        # 3. Soft Obstacles (Smoke)
        # Original: if v.type == 'SMOKE': cost += 5.0
        if v.type == 'SMOKE': 
            cost += 50.0  # Cranked up from 5.0! The agent will now treat smoke almost like a wall.

```

*Effect:* The agent will take massive detours to avoid smoke, even if it means walking into a dense crowd.

---

### Experiment 3: The "Introverted" Agent (High Crowd Aversion)

If you want your agent to prefer taking the "long way around" rather than pushing through the mob, we can tweak the crowd logic.

Look for the **5. Crowd Logic** section in `c(self, u, v)`:

```python
        # 5. Crowd Logic
        if v.type == 'CROWD':
            active_exits = [n for n in (self.goals + self.windows) if n.type != 'RUBBLE']
            if not active_exits:
                dist = float('inf')
            else:
                dist = min([math.hypot(v.x - g.x, v.y - g.y) for g in active_exits])
            
            # Original: 
            # if dist < 3: cost += 2.0 
            # else: cost += 20.0 
            
            # NEW: Massive penalty for ANY crowd, regardless of how close to the exit.
            cost += 40.0 * v.crowd_density # Scales linearly with how dense the crowd is!

```

*Effect:* By scaling the cost dynamically with `v.crowd_density`, the algorithm maps dense clusters of people as literal mountains on the cost graph. The agent will weave through sparse areas to avoid the bottleneck.

---
Implementing dynamic edge weights for the crowd is a classic way to handle moving bottlenecks without breaking the graph. If you just treat the crowd as "walls" (infinite cost), the algorithm might conclude there is no valid path and the agent gets stuck.

By scaling the penalty based on `v.crowd_density`, you are essentially terrain-mapping the grid. A dense crowd becomes a high-cost "mountain" on the graph, while a sparse crowd is just a "hill." The D* Lite relaxation step will naturally start preferring longer geometric paths if the edge weights through the crowd are too heavy.

Here is the exact modification you need to make in **Cell 2**.

Find the `c(self, u, v)` function and replace the entire `# 5. Crowd Logic` block with this:

```python
        # 5. Crowd Logic (Density Scaling)
        if v.type == 'CROWD':
            # Instead of a flat penalty, we scale dynamically.
            # v.crowd_density represents how many people are adjacent to this cell.
            base_penalty = 10.0
            
            # Non-linear scaling: We square the density to make dense clusters 
            # disproportionately expensive compared to single individuals.
            # Example: 1 person = +10 cost. 3 people = +90 cost.
            cost += base_penalty * (v.crowd_density ** 2) 
            
            # Optional: Keep the exit proximity logic so the agent is still 
            # willing to push through a crowd if it's literally AT the exit door.
            active_exits = [n for n in (self.goals + self.windows) if n.type != 'RUBBLE']
            if active_exits:
                dist_to_exit = min([math.hypot(v.x - g.x, v.y - g.y) for g in active_exits])
                if dist_to_exit < 2:
                    cost -= 15.0 # Forgive some of the penalty if we are touching the exit

```

### What to watch for when you run this:

1. **The "Weaving" Behavior:** Instead of walking in a straight line through a group of 3-4 people, you'll see the stick figure take distinct jagged steps to thread the needle between less crowded cells.
2. **Replanning Spikes:** Since the crowd is constantly moving (updating `v.crowd_density` every step), the edge weights are highly volatile. You might notice your replanning events (`self.km`) at the end of the simulation go up slightly because the graph is constantly shifting beneath the agent.
3. **The Bottleneck Push:** Notice the `dist_to_exit` override at the bottom. This ensures that if the *only* way out is through a mob gathered at the door, the agent will still push through rather than pacing back and forth in the middle of the room.

Swap that block into **Cell 2**, run it, and then run **Cell 4** to see it in action.


Taking a project from a Jupyter Notebook prototype to a production-grade system is a huge and rewarding leap. To make this a "real" project—something that solves a tangible problem and stands out technically—you need to move away from randomized grids and slow rendering loops, and start thinking about performance, real-world data, and system architecture.

Here is a roadmap to transform this prototype into a robust, real-world evacuation system:

### 1. Upgrade the Engine: Porting to C++

Python and Matplotlib are fantastic for testing logic, but they are bottlenecks for large-scale, real-time simulations.

* **The Change:** Rewrite the core D* Lite algorithm and grid management in C++ to handle thousands of nodes and agents with microsecond latency.
* **Integration:** You don't have to abandon Python entirely. You can compile the C++ code into a shared library and bind it to Python using `pybind11`, or set it up as a backend service that communicates with a frontend via WebSockets or a REST API.

### 2. Real-World Environments (No More Random Walls)

A real evacuation system needs to operate on real floor plans, not randomly generated blocks.

* **The Change:** Build a module that ingests actual building blueprints. You can take image files of campus buildings or office floors and use computer vision techniques (like contour detection via OpenCV) to automatically parse walls, rooms, and doors into your grid or a Navigation Mesh (NavMesh).

### 3. Advanced Crowd Physics: The Social Force Model

Currently, your crowd agents move somewhat randomly. Real crowds behave like fluid dynamics under pressure.

* **The Change:** Implement the **Social Force Model** for your crowd agents. This treats people like particles affected by physical forces. Agents have an internal desire to move toward an exit (), but experience repulsive forces from walls () and other people ().
* **The Math:** This allows you to simulate realistic panics, arching at doorways, and crushing at bottlenecks. The core equation looks like this:



### 4. Predictive AI & Trajectory Modeling

Instead of just reacting to the crowd, the system could predict where danger will occur.

* **The Change:** Train a sequence model (like a Mamba SSM or an LSTM) on historical crowd evacuation datasets. Use it to analyze the real-time movement of the crowd and predict where severe bottlenecks or "crush zones" will form *before* they happen, allowing your pathfinder to route the user away from future traffic jams.

### 5. Data Pipelines & Analytics

A real-world safety tool needs to log what happened for post-incident review.

* **The Change:** Treat your simulation like an IoT network. Set up a data pipeline where simulated "smoke detectors" and agent coordinates stream data into a robust database like PostgreSQL.
* **Analytics:** Build queries to analyze the data post-evacuation. Calculate the average time to escape, highlight structural bottlenecks, and visualize the spread of the fire over time.

### 6. Decoupled Interface

Matplotlib is a data visualization tool, not a simulation frontend.

* **The Change:** Build a dedicated UI. If you want a desktop app, Pygame is a great lightweight step up. If you want a dashboard, build a web frontend (using React and HTML5 Canvas or Three.js) that visualizes the data streaming from your backend.

---

To get started, it's usually best to tackle one architectural shift at a time. Would you like me to help you sketch out the C++ architecture for the D* Lite core, or would you rather look into writing a script to parse real floor plan images into a workable grid?

Totally fair! Let's dial it back. If this is a final-year or lab project, you want maximum visual and academic impact without having to learn a new stack or spend months debugging architecture.

We will keep it 100% Python and focus on features that will make your professors and lab evaluators really happy.

Here are four highly effective, pure-Python upgrades you can add to take this from a basic script to a standout lab project:

### 1. Image-Based Map Importer (using OpenCV)

Randomly generated walls are okay for testing, but loading a specific floor plan looks much more professional.

* **How it works:** Draw a simple map in MS Paint (Black = Walls, White = Floor, Green = Exits). Use OpenCV (`cv2.imread`) to read the image. Loop through the pixels, and if a pixel is black, set that grid node's type to `'WALL'`.
* **The Impact:** In about 15 lines of Python, you go from random blocks to simulating evacuations for your actual university building or lab layout.

### 2. The "A* vs. D* Lite" Benchmark

For an academic lab, evaluators love comparisons. D* Lite is literally designed to be better than A* in dynamic environments (like when a fire suddenly blocks a path).

* **How it works:** Add a standard A* algorithm to your notebook. Run the simulation twice on the same map.
* **The Impact:** You can show that when the fire spreads, A* has to recalculate the *entire* path from scratch (expanding hundreds of nodes), while your D* Lite only updates the local changes.

### 3. Automated Performance Metrics & Tables

Right now, the simulation just prints "Simulation Complete." To make this a proper project, you need hard data at the end.

* **How it works:** Track metrics like `total_steps_taken`, `computation_time_per_step`, and `crowd_density_encountered`. At the end of the run, use Pandas to generate a clean, formatted table of these performance metrics.
* **The Impact:** Having professionally formatted evaluation tables ready to drop straight into a presentation or a project report shows rigor. You can run the simulation 50 times in a loop (without the visualization) to find the "Average Escape Time" under different fire conditions.

### 4. Upgrade the Visuals (Pygame)

Matplotlib is great for data, but it's notoriously slow and clunky for real-time animations.

* **How it works:** Swap Matplotlib for **Pygame**. It's pure Python, incredibly easy to learn, and handles 2D grid drawing effortlessly at 60 frames per second.
* **The Impact:** Your simulation will go from looking like a choppy graphing calculator to a smooth, interactive piece of software.

---

Any of these will massively elevate the project without taking you out of Python. Would you like me to write the 15-line OpenCV snippet so you can start loading custom maps drawn in MS Paint?